<a href="https://colab.research.google.com/github/ZaxkyyOfficial/Flyrank-AI/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZaxkyyOfficial/Flyrank-AI/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*
My Rule:
Pages will be prioritized for refreshing if they are old (>180 days) AND previously generated significant traffic (>100 impressions) during the prior period. The maximum score is 100 (60 points for age, 40 points for traffic volume).

Reason Codes:
stale_visible_page

Signal Checks (Pre-requisites):

Signal 1: Staleness (content_age_days). Verdict: CONFIRMED. Older pages tend to experience higher rates of decline compared to newer pages.

Signal 2: Volume (impressions_prev15). Verdict: MIXED. Pages with very low volume are highly volatile, whereas high-volume pages are more stable but can still become stale.

In [3]:
import duckdb
import os
import pandas as pd

# 1. Setup Token & Koneksi DuckDB
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

# 2. Tarik Data (menggunakan content_created_date yang benar)
df = con.sql(f"""
    WITH early_march AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS impressions_prev15
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE report_date <= '2026-03-15'
        GROUP BY 1, 2
    ),
    late_march AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS impressions_next15
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE report_date > '2026-03-15'
        GROUP BY 1, 2
    )
    SELECT e.client_hash_id, e.content_hash_id, e.impressions_prev15,
           COALESCE(l.impressions_next15, 0) AS impressions_next15,
           -- PERBAIKAN: Menggunakan content_created_date
           date_diff('day', CAST(d.content_created_date AS DATE), CAST('2026-03-15' AS DATE)) AS content_age_days,
           CASE WHEN COALESCE(l.impressions_next15, 0) < (0.8 * e.impressions_prev15) THEN 1 ELSE 0 END AS is_declining
    FROM early_march e
    LEFT JOIN late_march l ON e.client_hash_id = l.client_hash_id AND e.content_hash_id = l.content_hash_id
    LEFT JOIN read_parquet('{REL}/dim_content.parquet') d ON e.content_hash_id = d.content_hash_id
    WHERE e.impressions_prev15 >= 50
""").df()

df['content_age_days'] = df['content_age_days'].fillna(0)

# 3. Check Two Signals (Bukti dari Verdict di atas)
print("=== SIGNAL 1: STALENESS (Content Age) ===")
df['age_tier'] = pd.cut(df['content_age_days'], bins=[-1, 90, 180, 365, 9999], labels=['0-3mo', '3-6mo', '6-12mo', '>1yr'])
display(df.groupby('age_tier', observed=True)['is_declining'].agg(['count', 'mean']).rename(columns={'count': 'n', 'mean': 'decline_rate'}))

print("\n=== SIGNAL 2: VOLUME (Impressions) ===")
df['volume_tier'] = pd.qcut(df['impressions_prev15'], q=4, labels=['Low', 'Med-Low', 'Med-High', 'High'])
display(df.groupby('volume_tier', observed=True)['is_declining'].agg(['count', 'mean']).rename(columns={'count': 'n', 'mean': 'decline_rate'}))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== SIGNAL 1: STALENESS (Content Age) ===


,n,decline_rate
age_tier,,
0-3mo,30187,0.294531
3-6mo,18190,0.384277
6-12mo,34593,0.246726
>1yr,9578,0.218522



=== SIGNAL 2: VOLUME (Impressions) ===


,n,decline_rate
volume_tier,,
Low,23208,0.291580
Med-Low,23115,0.291413
Med-High,23098,0.273054
High,23127,0.289661


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*
I will calculate the `baseline_score` statically based on the rules above. The data will then be sorted by the highest score (top priority), and the results will be exported to the `baseline_action_score.csv` file.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Menerapkan Aturan Baseline (Skor Maksimal 100)
df['baseline_score'] = 0
df.loc[df['content_age_days'] > 180, 'baseline_score'] += 60
df.loc[df['impressions_prev15'] > 100, 'baseline_score'] += 40

# Menambahkan Reason Code dan Action
df['reason_code'] = 'stale_visible_page'
df['action'] = 'Review for Refresh'

# Mengurutkan berdasarkan skor tertinggi, lalu impresi tertinggi sbg tie-breaker
df_ranked = df.sort_values(by=['baseline_score', 'impressions_prev15'], ascending=[False, False]).reset_index(drop=True)

# Membuat direktori dan menyimpan CSV (Sesuai instruksi: out of git)
os.makedirs('work/outputs', exist_ok=True)
csv_path = 'work/outputs/baseline_action_score.csv'
df_ranked.to_csv(csv_path, index=False)

print(f"File antrean prioritas berhasil disimpan di: {csv_path}")
print(f"Total halaman dalam antrean: {len(df_ranked):,}")

File antrean prioritas berhasil disimpan di: work/outputs/baseline_action_score.csv
Total halaman dalam antrean: 92,548


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*
Top-10 Review:

Action: Review for Refresh | Reason: stale_visible_page | Incorrect if: The decline is purely due to seasonality.

Action: Review for Refresh | Reason: stale_visible_page | Incorrect if: A competitor page from the client's own domain is cannibalizing this page's traffic.

Action: Review for Refresh | Reason: stale_visible_page | Incorrect if: The primary search query is trending downwards globally.

Action: Review for Refresh | Reason: stale_visible_page | Incorrect if: A temporary technical issue (server down/500 error) caused a drop in impressions.

Action: Review for Refresh | Reason: stale_visible_page | Incorrect if: The page was revised just a few days ago but is not yet fully indexed by Google.

Action: Review for Refresh | Reason: stale_visible_page | Incorrect if: A UI design change caused a drastic drop in CTR, while the text content itself remains relevant.

Action: Review for Refresh | Reason: stale_visible_page | Incorrect if: The product/service discussed in the article is no longer sold by the client.

Action: Review for Refresh | Reason: stale_visible_page | Incorrect if: A temporary Google algorithm fluctuation occurred; traffic could return to normal without editing.

Action: Review for Refresh | Reason: stale_visible_page | Incorrect if: The page was moved (URL changed without a proper 301 redirect).

Action: Review for Refresh | Reason: stale_visible_page | Incorrect if: Client Search Console data was interrupted for several days, skewing the average calculations.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Menampilkan 10 baris teratas dari antrean yang sudah di-ranking
display(df_ranked[['content_hash_id', 'baseline_score', 'reason_code', 'action', 'impressions_prev15']].head(10))

,content_hash_id,baseline_score,reason_code,action,impressions_prev15
0,content_eadb33b5df496f4a,100,stale_visible_page,Review for Refresh,161575.0
1,content_e8a52cf3d5988c07,100,stale_visible_page,Review for Refresh,143173.0
2,content_ec2e0346994fb5a5,100,stale_visible_page,Review for Refresh,132811.0
3,content_36e53e9c707674fc,100,stale_visible_page,Review for Refresh,109909.0
4,content_e7b5dd4dff461ad2,100,stale_visible_page,Review for Refresh,87662.0
5,content_3df3f32f3fd58dea,100,stale_visible_page,Review for Refresh,84041.0
6,content_9c057b66c30a3abb,100,stale_visible_page,Review for Refresh,83772.0
7,content_471d9cabce329a66,100,stale_visible_page,Review for Refresh,79546.0
8,content_fd2117c2c6790e4b,100,stale_visible_page,Review for Refresh,78162.0
9,content_34a70fea29d15f24,100,stale_visible_page,Review for Refresh,73639.0


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*
Weak Picks:
Because this relies on a fixed rule, the model assigns an identical score of "100" to thousands of pages simultaneously (based solely on the criteria of being older than 180 days and having over 100 impressions). This is a major drawback of static rules; the content team cannot distinguish the most urgent priorities among the thousands of pages sharing that score of 100.

Leakage Check:
I have confirmed that the score (`baseline_score`) is calculated PURELY using historical data (`content_age_days` and `impressions_prev15`). No product flags (such as FlyRank's built-in `priority_score`) or future-looking metrics (like `impressions_next15`) have leaked into the rule creation process.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verifikasi otomatis untuk memastikan tidak ada kolom bocor yang dipakai untuk menghitung skor
used_columns = ['content_age_days', 'impressions_prev15']
leaked_columns = ['impressions_next15', 'is_declining', 'priority_score', 'health_score']

for col in leaked_columns:
    assert col not in used_columns, f"BAHAYA LEAKAGE: Kolom '{col}' dipakai untuk menyusun baseline rule!"

print("Leakage Check AMAN: Aturan statis murni dibangun dari data masa lalu.")

Leakage Check AMAN: Aturan statis murni dibangun dari data masa lalu.


## Self-check

Before you submit, confirm each line honestly:

- [x] Two signal verdicts with visible bucket tables and n
- [x] One rule with a score, a reason code, and an action label
- [x] Ranked queue written to work/outputs/baseline_action_score.csv
- [x] Ten reviewed rows with "what would make it wrong"
- [x] No future-window or label-derived inputs used in the rule
- [x] Committed to my repo under work/notebooks/